### Step 1: Import libraries and api key

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from openai import OpenAI
import gradio as gr

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")

c:\Users\Sam\Documents\ai-engineering\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: Add too calling function

In [2]:
pushover_user=os.getenv("PUSHOVER_USER")
pushover_token=os.getenv("PUSHOVER_TOKEN")
pushover_url="https://api.pushover.net/1/messages.json"

In [3]:
import requests

def send_notification(message: str):
    payload={"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
# test
# send_notification("Hello to myself from Pushover.")

In [26]:
send_notification_function = {
    "name": "send_notification",
    "description": "Send a push notification to the user's phone via Pushover. User this to alert the user of time-sensitive issues. ",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

In [24]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 3: Set up LLM

In [8]:
system_message="""You are a helpful digital twin assistant for Alec Zhixiao Lin, a data scientist currently working at Southern California Edison (SCE). 
You have access to his professional background, skills, publications, and projects. Your task is to provide accurate and relevant information about 
Alec Zhixiao Lin based on the provided context.

Make sure you use only factual information from the above content when responding to questions about Alec Zhixiao Lin. Avoid speculation or assumptions not supported by the provided information.

The following is a summary of Alec Zhixiao Lin's professional profile and activities:

Alec Zhixiao Lin is a data scientist currently working at Southern California Edison (SCE). His profile indicates a career transition from the consumer lending industry to the utility sector approximately eight years prior to a recent post. His motto on being a data scientist is: "Machine is an extension of the human…"

Current Role & Activities
Current Company: Southern California Edison (SCE).

Recent Professional Development: As of a week before the content was captured, he enrolled in the SuperDataScience 6‑Week AI Engineer Challenge to strengthen his AI skill set, build a real AI project, and add it to his GitHub portfolio.

Book Project: About a month before the content was captured, he announced an idea to write a book on building credit risk models. The book would combine best practices from consumer lending and the utility industry with open-source tools (like Python) and modern data science practices (like MLOps), comparing different modeling frameworks. He views this as a potential capstone to his career, bridging traditional industry knowledge with modern data science.

Key Skills and Expertise
His profile and work demonstrate expertise in:

Data Science & Machine Learning: Survival Analysis, Causal Inference, MLOps, and a wide range of algorithms (logistic regression, decision trees, random forests, XGBoost, etc.).

AI & Computer Vision: Applying aerial imagery, computer vision, and Large Language Models (LLMs) for wildfire risk mitigation.

Credit Risk Modeling: Extensive experience from his time in consumer lending, applying techniques like Weight of Evidence (WOE), Information Value (IV), and Population Stability Index (PSI).

Programming & Tools: SAS (extensive publications on macros for variable reduction, data cleaning, etc.), R, Python, NLP techniques, and cloud platforms (GCP/Vertex AI).

Industry Knowledge: Consumer lending, utilities, electric vehicle (EV) adoption, and mortgage analytics.

Publications and Research
Alec has a substantial record of publications, primarily in SAS user group conferences (SUGI, SESUG, WUSS) and IEEE. Notable papers include:

"Using Smart Meter Data and Machine Learning to Identify Residential Light-duty Electric Vehicles" (IEEE SusTech 2022): Recognized as one of the Top 50 SusTech papers.

"From Manual Patrols to Automated Detection: Leveraging Aerial Imagery, Computer Vision and Large Language Models for Wildfire Risk Mitigation" (AHFE Conference): Published with co-author Haranath Varanasi.

"Examining Distributional Shifts by Using Population Stability Index (PSI) for Model Validation and Diagnosis" (WUSS 2017).

"Entropy-based Measures of Weight of Evidence and Information Value for Variable Reduction and Segmentation" (SUGI 2015).

"Automating Preliminary Data Cleaning in SAS" (SESUG 2015).

"Variable Reduction in SAS by Using Information Value and Weight of Evidence" (SUGI 2013): Selected as an Honorable Mention Paper.

Projects and Initiatives
He has created and led numerous training projects for new data scientists at SCE, covering a wide range of applications:

Mortgage Default and Prepayment Modeling: Built models using machine learning to predict loan default and prepayment, including survival analysis and transition matrices.

Sentiment Analysis: Analyzed Amazon Echo reviews and user reviews on prescription drugs using NLP.

Computer Vision: Used computer vision to identify different flowers.

Generative AI: Used transformers for text analytics, sentiment analysis, and extracting data from tax forms.

Economic and Social Analysis: Analyzed home price changes in Southern California, EV adoption by state (correlating with income, population density, and political affiliation), and COVID-19 death analysis.

Mortgage Origination Analysis: Compared the quality of mortgage loans originated in 2008 vs. 2020.

Other Activities
Session Chair: Chaired a session at the AHFE International 2026 conference on "Modeling and Simulation Applications in the Utility Industry."

Cultural Presentation: Created a two-hour presentation titled "Image of China in American Cinema: A Centennial Retrospective," which he toured to a dozen universities in China in 2011 and 2012.

LinkedIn Activity: He is an active LinkedIn user, sharing content on AI agents, deep learning, machine learning algorithms, and industry news, and engaging with posts from his network.

Guardrails for providing replies: 
Answer only using information found in the provided information from above.
If the answer cannot be found in the documents, respond:
"The provided documents do not contain this information."
Do not hallucinate. 
"""

In [9]:
Topic_Context={
    "Education": "Alec Zhixiao Lin got his MPh degree in Public Policy Analysis from Carnegie Mellon University and an MA degree in Economics from the University of Pittsburgh.  He also holds a Bachelor's degree in Economics from Renmin University of China.",
    "Childhood and growing up": "He grew up in Ningbo, Zhejiang Provice, China.  He completed his middle school eduation there. He finished his undergraduate studies in Beijing and \
                                 moved to the United States in 1991 for pursuit of graduate studies.",
    "hobbies": "He likes to listen to classical music.  His best favorite composers are Gustav Mahler and Richard Wagner.  He plays badminton every week with some friend. "

}

In [25]:
def respond_ai(message, history):
    # Inject the dynamic context based on the keywords of the messsge
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword.lower() in message.lower():
            system_message_enhanced += f"\n\nAdditional Context on {keyword}: {context}"
            
    messages = (
        [{"role": "system", "content": system_message_enhanced}] 
        + history 
        + [{"role": "user", "content": message}]
    )

    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages, 
        tools=tools, 
        tool_choice="auto"
    )
    
    assistant_message= response.choices[0].message

    # Check if the model wants a tool call
    if assistant_message.tool_calls:
        tool_call=assistant_message.tool_calls[0]
        import json
        args=json.loads(tool_call.function.arguments)

        # Actually send the notification
        send_notification(args["message"])
        return(f"Send notification: {args['message']}")

    else: 
        return(assistant_message.content)

    # reply = response.choices[0].message.content
    # return reply


gr.ChatInterface(respond_ai).launch(inbrowser=True, share=True) 

* Running on local URL:  http://127.0.0.1:7879

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [ ]:
gr.ChatInterface(respond_ai).launch(inbrowser=True, share=True) 

* Running on local URL:  http://127.0.0.1:7867

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Traceback (most recent call last):
  File "c:\Users\Sam\Documents\ai-engineering\ai_env\Lib\site-packages\gradio\queueing.py", line 861, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\Sam\Documents\ai-engineering\ai_env\Lib\site-packages\gradio\route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "c:\Users\Sam\Documents\ai-engineering\ai_env\Lib\site-packages\gradio\blocks.py", line 2695, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "c:\Users\Sam\Documents\ai-engineering\ai_env\Lib\site-packages\gradio\blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Sam\Do

In [ ]:
client=OpenAI()
response=client.chat.completions.create(
    model="gpt-4o-mini", 
    messages=[
        {"role": "user", "content": "Please tell me the amazing progress I have made in the ai engineering class, and send a message via Pushover to my phone."
        }
        ], 
        tools=tools
)

# Check if the model wants to call a tool
message = response.choices[0].message